# Detections

So far, just looking through all of the long observations hasn't yielded any flares. It also hasn't yielded any emission at all, but we know these stars did emit, at least once, because they are in the SRSC. What if we looked at the observations where they WERE detected and see what sort of emission they produced.

In [42]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.table import vstack
from astroquery.gaia import Gaia
import tqdm
from astropy.table import join
from scipy.optimize import root_scalar
from astroquery.simbad import Simbad
from ipyaladin import Aladin
from astroquery.casda import Casda
from astropy.coordinates import SkyCoord
from astroquery.utils.tap.core import TapPlus
import astropy.units as u
import anywidget
import ipywidgets
import ipyaladin
import os
import re
import pandas as pd
from pathlib import Path

In [2]:
OPAL_USER = 'jfer9940@uni.sydney.edu.au'
Casda.login(username=OPAL_USER, store_password=True)

INFO: Authenticating jfer9940@uni.sydney.edu.au on CASDA ... [astroquery.casda.core]
INFO: Authentication successful! [astroquery.casda.core]


First, we need a list of sources we are interested in looking at. This is our sample of ~40 stars. We also need a table of the detections themselves.

In [33]:
sources = Table.read('data/source_obs_data.csv')
observations = Table.read('data/2026.01.16_12_07_51_SRSC_Var_alldetections.csv')
source_identifiers = Table.read('data/2026.01.16_12_07_51_SRSC_Static.csv')

In [5]:
sources

Simbad_ID,Archival_ra,Archival_dec,n_obs,survey_names
str19,float64,float64,int64,str67
HD 102982,177.787504285457,-51.876071495961,2,"FLASH, WALLABY"
V* V376 Cep,329.577295376914,82.870902021925,0,--
V* V815 Her,272.06734740577,29.6910259723,0,--
HD 165590A,271.457229,21.44599,0,--
HD 82533,142.67769359508,-53.521472152447,3,"EMU, GASKAP-HI"
HD 217344,345.11766870906,-33.745748964489,22,"DINGO, FLASH, WALLABY"
HD 8558,20.839329303534,-57.480907517274,2,WALLABY
HD 200244,315.742708658278,-22.933914165698,2,FLASH
HD 162186,267.778490987213,-32.307805803789,10,An ASKAP Survey for Galactic Radio Transients in the Baade's Window


In [6]:
observations

index,Identifier,Search_method,Crossmatch_radius_arcsec,Radio_optical_sep_arcsec,Radio_component_id,Radio_SBID,Radio_starttime_mjd,Radio_endtime_mjd,Radio_starttime_isot,Radio_endtime_isot,Radio_exptime_s,Radio_ra,Radio_dec,Radio_ra_err,Radio_dec_err,Radio_ra_hms,Radio_dec_dms,Radio_freq_MHz,Radio_I_flux_peak,Radio_I_flux_peak_err,Radio_I_flux_int,Radio_I_flux_int_err,Radio_maj_axis,Radio_min_axis,Radio_pos_angle,Radio_maj_axis_err,Radio_min_axis_err,Radio_survey,Radio_I_measurementType,Radio_V_measurementType,Radio_V_flux_peak,Radio_V_flux_peak_err,Radio_V_flux_int,Radio_V_flux_int_err,Radio_telescope,Radio_local_V_RMS,Radio_local_I_RMS,Reference,unique_obs_id
int64,str10,str15,float64,float64,str37,str185,float64,float64,str23,str23,str18,float64,float64,float64,float64,str17,str18,str13,float64,float64,float64,float64,float64,float64,float64,float64,float64,str16,str13,str13,float64,float64,float64,float64,str19,float64,str18,str153,str185
1899,SRSC 00000,LightCurve,2.0,1.1,SB22350_component_2094a,SB22350_RACS_1424-60,59246.87228009,59246.88263889,2021-02-01T20:56:05.000,2021-02-01T21:11:00,895.0,216.7673,-62.7821,2.0,2.0,14:27:04.1,-62:46:55,1367.5,1.6,0.3,1.6,0.5,9.0,8.0,170.91,2.0,1.0,--,Detection,NoMeasurement,--,--,--,--,ASKAP,--,0.28,SRSCv1.0,SB22350_RACS_1424-60
1909,SRSC 00000,LightCurve,2.0,1.6,SB45665_component_4457a,SB45665_VAST_1404-62,59902.05111111,59902.05940972,2022-11-19T01:13:36.000,2022-11-19T01:25:33,717.0,216.767,-62.7823,2.0,2.0,14:27:04.1,-62:46:56,887.5,1.6,0.2,1.8,0.3,19.0,11.0,134.44,2.0,1.0,--,Detection,NoMeasurement,--,--,--,--,ASKAP,--,0.21,SRSCv1.0,SB45665_VAST_1404-62
1910,SRSC 00000,LightCurve,2.0,0.7,SB45667_component_4705a,SB45667_VAST_1453-62,59902.06976852,59902.07806713,2022-11-19T01:40:28.000,2022-11-19T01:52:25,717.0,216.7676,-62.7823,2.0,2.0,14:27:04.2,-62:46:56,887.5,1.4,0.2,1.7,0.4,16.0,15.0,86.21,2.0,2.0,--,Detection,NoMeasurement,--,--,--,--,ASKAP,--,0.24,SRSCv1.0,SB45667_VAST_1453-62
1914,SRSC 00000,ROSAT_Stellar,2.8,0.7,SB45821_component_4152b,SB45821_EMU_1442-64,59907.98746528,59908.40425926,2022-11-24T23:41:57.000,2022-11-25T09:42:08,36011.0,216.7681,-62.7819,2.0,2.0,14:27:04.3,-62:46:55,943.5,0.64,0.03,0.74,0.05,17.5,15.0,133.57,0.7,0.7,10-11hour,Detection,NoMeasurement,--,--,--,--,ASKAP,--,0.03,SRSCv1.0,SB45821_EMU_1442-64
1912,SRSC 00000,LightCurve,2.0,2.2,SB47236_component_4433a,SB47236_VAST_1404-62,59964.86481481,59964.87321759,2023-01-20T20:45:20.000,2023-01-20T20:57:26,726.0,216.7684,-62.7827,2.0,2.0,14:27:04.4,-62:46:58,887.5,1.4,0.2,1.7,0.4,20.0,11.0,135.82,2.0,2.0,--,Detection,NoMeasurement,--,--,--,--,ASKAP,--,0.24,SRSCv1.0,SB47236_VAST_1404-62
1913,SRSC 00000,StokesV,--,--,SB47240_component_3629a,SB47240_VAST_1453-62,59964.90674769,59964.90674769,2023-01-20T21:45:43.000,2023-01-20T21:45:43,726.0,216.7677,-62.783,--,--,14:27:04.24,-62:46:59,887.5,2.1,0.2,2.8,0.4,--,--,--,--,--,VAST 23e,Detection,Detection,1.3,--,1.9,--,ASKAP,--,0.29,10.1093/mnras/stab299,SB47240_VAST_1453-62
1908,SRSC 00000,LightCurve,2.0,1.2,SB50301_component_4182a,SB50301_VAST_1453-62,60097.53744213,60097.54572917,2023-06-02T12:53:55.000,2023-06-02T13:05:51,716.0,216.7685,-62.7823,2.0,2.0,14:27:04.4,-62:46:56,887.5,1.6,0.3,1.5,0.5,17.0,11.0,123.01,3.0,2.0,--,Detection,NoMeasurement,--,--,--,--,ASKAP,--,0.26,SRSCv1.0,SB50301_VAST_1453-62
1911,SRSC 00000,ROSAT_Stellar,2.7,0.6,SB51091_component_4642a,SB51091_VAST_1453-62,60129.45053241,60129.45883102,2023-07-04T10:48:46.000,2023-07-04T11:00:43,717.0,216.7675,-62.7821,2.0,2.0,14:27:04.2,-62:46:56,887.5,1.3,0.3,1.2,0.5,15.0,11.0,164.89,3.0,2.0,00-01hour,Detection,NoMeasurement,--,--,--,--,ASKAP,--,0.25,SRSCv1.0,SB51091_VAST_1453-62
1906,SRSC 00000,LightCurve,2.0,2.5,SB51522_component_4617a,SB51522_VAST_1453-62,60146.41688657,60146.42528935,2023-07-21T10:00:19.000,2023-07-21T10:12:25,726.0,216.7664,-62.782,2.0,2.0,14:27:03.9,-62:46:55,887.5,1.4,0.2,1.6,0.4,22.0,10.0,141.85,2.0,2.0,--,Detection,NoMeasurement,--,--,--,--,ASKAP,--,0.25,SRSCv1.0,SB51522_VAS

In [7]:
source_identifiers

Identifier,Simbad_ID,GaiaDR3_ID,TYCHO_ID,2MASS_ID,GCVS_ID,HIP_ID,UCAC4_ID,Archival_Survey,Archival_ID,Archival_epoch,Archival_ra,Archival_ra_err,Archival_dec,Archival_dec_err,Archival_ra_hms,Archival_dec_dms,Archival_parallax,Archival_parallax_err,Archival_pmra,Archival_pmra_err,Archival_pmdec,Archival_pmdec_err,Radio_multiple,Search_method_bin,1RXS_ID,TIC_ID,WISE_ID,AllWISE_ID,CNS5_ID,Stripped_Identifier,Stripped_Archival_ID
str10,str31,str28,str15,str23,str14,str10,str16,str39,str28,str19,float64,float64,float64,float64,str17,str19,float64,float64,float64,float64,float64,float64,float64,int64,str21,str14,str25,str25,str9,str9,str26
SRSC 00000,CPD-62 4126,Gaia DR3 5853502492753436416,TYC 9010-2749-1,2MASS J14270430-6246553,--,--,UCAC4 137-115021,GaiaDR3,Gaia DR3 5853502492753436416,J2016,216.767919155244,6e-06,-62.782098179545,9e-06,14:27:04,-62:46:55.553446362,1.53,0.01,-3.841,0.007,-5.44,0.01,0.0,18,1RXS J142704.5-624659,TIC 385279385,--,--,--,SRSC00000,GaiaDR35853502492753436416
SRSC 00001,TYC 9078-815-1,Gaia DR3 6438561254116213760,TYC 9078-815-1,2MASS J19040331-6400142,--,--,UCAC4 130-173182,GaiaDR3,Gaia DR3 6438561254116213760,J2016,286.013875552226,8e-06,-64.00397308781,9e-06,19:04:03,-64:00:14.303116116,2.38,0.01,6.74,0.01,-4.68,0.01,0.0,18,1RXS J190403.3-640011,--,--,--,--,SRSC00001,GaiaDR36438561254116213760
SRSC 00002,SCR J2241-6119A,Gaia DR3 6406967509044836096,--,2MASS J22414436-6119311,--,--,UCAC4 144-211738,GaiaDR3,Gaia DR3 6406967509044836096,J2016,340.436272395047,8e-06,-61.325706059717,1e-05,22:41:45,-61:19:32.541814980,35.2,0.01,150.4,0.01,-87.89,0.01,0.0,18,--,TIC 232064182,WISE J224144.56-611932.0,--,--,SRSC00002,GaiaDR36406967509044836096
SRSC 00003,CD-30 6530,Gaia DR3 5640708647248169600,TYC 7136-2264-1,2MASS J08355977-3042306,--,--,UCAC4 297-048519,GaiaDR3,Gaia DR3 5640708647248169600,J2016,128.998712548091,8e-06,-30.70858670455,1e-05,08:36:00,-30:42:30.91213638,15.7,0.01,-64.11,0.01,-14.3,0.01,0.0,18,1RXS J083559.7-304226,TIC 11663272,--,--,--,SRSC00003,GaiaDR35640708647248169600
SRSC 00004,CD-49 451,Gaia DR3 4917782741272766592,TYC 8043-814-1,2MASS J01372081-4911443,--,HIP 7554,UCAC4 205-001600,GaiaDR3,Gaia DR3 4917782741272766592,J2016,24.34012397651,1e-05,-49.195143947004,1e-05,01:37:22,-49:11:42.51820921,45.12,0.01,496.1,0.01,116.1,0.01,0.0,18,1RXS J013721.1-491144,TIC 158624542,--,WISEA J013721.34-491143.1,CNS5 406,SRSC00004,GaiaDR34917782741272766592
SRSC 00005,ATO J183.5330+47.2673,Gaia DR3 1545247048904161664,--,2MASS J12140814+4716038,--,--,UCAC4 687-053931,GaiaDR3,Gaia DR3 1545247048904161664,J2016,183.532979271787,1e-05,47.267364915243,1e-05,12:14:08,+47:16:02.51369487,25.76,0.02,-137.2,0.01,-72.58,0.01,0.0,16,--,TIC 138182259,--,WISEA J121407.98+471602.9,--,SRSC00005,GaiaDR31545247048904161664
SRSC 00006,HD 102982,Gaia DR3 5368777294028600448,TYC 8228-2739-1,2MASS J11510909-5152323,--,--,UCAC4 191-064625,GaiaDR3,Gaia DR3 5368777294028600448,J2016,177.787504285457,1e-05,-51.876071495961,1e-05,11:51:09,-51:52:33.85738546,18.8,0.02,-57.34,0.02,-83.67,0.01,0.0,18,1RXS J115108.5-515223,TIC 94637805,--,--,--,SRSC00006,GaiaDR35368777294028600448
SRSC 00007,2MASS J11414215-6521298,Gaia DR3 5236539580341818624,--,2MASS J11414215-6521298,--,--,UCAC4 124-048667,GaiaDR3,Gaia DR3 5236539580341818624,J2016,175.425221260026,1e-05,-65.358320600806,1e-05,11:41:42,-65:21:29.95416290,9.92,0.02,-40.18,0.02,-4.38,0.02,0.0,18,1RXS J114142.1-652128,TIC 321772880,--,--,--,SRSC00007,GaiaDR35236539580341818624
SRSC 00008,Gaia DR3 6090676737160646528,Gaia DR3 6090676737160646528,--,--,--,--,--,GaiaDR3,Gaia DR3 6090676737160646528,J2016,211.790382522137,1e-05,-50.034311567705,1e-05,14:07:10,-50:02:03.52164374,10.99,0.02,-54.59,0.02,-46.85,0.02,0.0,16,--,--,--,--,--,SRSC00008,GaiaDR36090676737160646528


First, to get only the observations for our sources, we need to find the SRSC identifier for each of our sources

In [4]:
source_list = list(sources['Simbad_ID'])

In [5]:
source_identifiers = [source_identifiers['Identifier'][source_identifiers['Simbad_ID'] == source][0] for source in source_list]

In [6]:
source_obs = vstack([observations[observations['Identifier'] == ident] for ident in source_identifiers])

Next, since we are only working with ASKAP data here, let's just remove any other observations

In [7]:
source_obs_askap = source_obs[source_obs['Radio_telescope'] == 'ASKAP']

In [12]:
source_obs_askap

index,Identifier,Search_method,Crossmatch_radius_arcsec,Radio_optical_sep_arcsec,Radio_component_id,Radio_SBID,Radio_starttime_mjd,Radio_endtime_mjd,Radio_starttime_isot,Radio_endtime_isot,Radio_exptime_s,Radio_ra,Radio_dec,Radio_ra_err,Radio_dec_err,Radio_ra_hms,Radio_dec_dms,Radio_freq_MHz,Radio_I_flux_peak,Radio_I_flux_peak_err,Radio_I_flux_int,Radio_I_flux_int_err,Radio_maj_axis,Radio_min_axis,Radio_pos_angle,Radio_maj_axis_err,Radio_min_axis_err,Radio_survey,Radio_I_measurementType,Radio_V_measurementType,Radio_V_flux_peak,Radio_V_flux_peak_err,Radio_V_flux_int,Radio_V_flux_int_err,Radio_telescope,Radio_local_V_RMS,Radio_local_I_RMS,Reference,unique_obs_id
int64,str10,str15,float64,float64,str37,str185,float64,float64,str23,str23,str18,float64,float64,float64,float64,str17,str18,str13,float64,float64,float64,float64,float64,float64,float64,float64,float64,str16,str13,str13,float64,float64,float64,float64,str19,float64,str18,str153,str185
1935,SRSC 00006,LightCurve,4.0,0.8048691716529892,SB21972_component_2117a,SB21972_RACS_1143-51,59239.80642361111,59239.81690972222,--,--,905.9999999357389,177.787726,-51.876138,0.33,0.35,--,--,1367.5,1.418,0.114,1.77,0.23,10.99,8.88,156.86,0.82,0.77,--,Detection,--,--,--,--,--,ASKAP,--,0.152,SRSCv1.1,SB21972_RACS_1143-51
1936,SRSC 00006,ROSAT_Stellar,3.9,1.1,SB50099_component_6783a,SB50099_WALLABY_1142-53A,60084.40834491,60084.74173611,2023-05-20T09:48:01.000,2023-05-20T17:48:06,28805.0,177.787,-51.876,2.0,2.0,11:51:08.9,-51:52:34,1367.5,0.32,0.02,0.42,0.05,10.3,8.5,1.63,0.7,0.6,08-09hour,Detection,NoMeasurement,--,--,--,--,ASKAP,--,0.03,SRSCv1.0,SB50099_WALLABY_1142-53A
1937,SRSC 00006,LightCurve,4.0,1.0316184965026012,SB64834_component_3847a,SB64834_VAST_1141-50,60538.28136574074,60538.28989583333,--,--,737.0000000344589,177.787561,-51.876043,0.52,0.56,--,--,887.5,1.99,0.185,2.311,0.356,14.18,12.81,148.8,1.37,1.15,--,Detection,--,--,--,--,--,ASKAP,--,0.225,SRSCv1.1,SB64834_VAST_1141-50
1938,SRSC 00006,LightCurve,4.0,0.6795587747548005,SB72938_component_4721a,SB72938_WALLABY_1142-53B,60781.48490740741,60781.81841435185,--,--,28814.99999994412,177.787072,-51.876434,0.27,0.29,--,--,1367.5,0.491,0.042,0.462,0.07,7.79,7.5,31.16,0.69,0.61,--,Detection,--,--,--,--,--,ASKAP,--,0.039,SRSCv1.1,SB72938_WALLABY_1142-53B
3180,SRSC 00145,LightCurve,4.0,3.418682076700671,RACS_1821+31A_component_125b,SB8548_RACS_1821+31A,58595.46135416667,58595.91984953704,--,--,39613.99999971036,272.067191,29.691917,0.21,0.55,--,--,887.5,2.203,0.051,4.311,0.238,52.61,17.76,179.76,1.68,1.11,--,Detection,--,--,--,--,--,ASKAP,--,0.37,SRSCv1.1,SB8548_RACS_1821+31A
3181,SRSC 00145,LightCurve,4.0,2.497374099970466,SB38571_component_4709a,SB38571_RACS_1821+31,59663.89630787037,59663.906793981485,--,--,906.0000005643816,272.068049,29.691493,0.48,0.98,--,--,887.5,1.372,0.083,3.108,0.26,45.17,15.53,172.86,2.31,1.1,--,Detection,--,--,--,--,--,ASKAP,--,0.262,SRSCv1.1,SB38571_RACS_1821+31
3408,SRSC 00175,ROSAT_Stellar,3.2,2.4,RACS_1811+18_2505,SB21623_RACS_1811+18,59234.09381803,59234.10418603,2021-01-20T02:15:05.000,2021-01-20T02:30:01,896.0,271.4569,21.4464,2.0,2.0,18:05:49.7,+21:26:47,1367.5,1.2,0.2,1.1,0.3,19.0,8.7,0.96,4.0,0.8,RACS-MID,Detection,NoMeasurement,--,--,--,--,ASKAP,--,--,SRSCv1.0,SB21623_RACS_1811+18
3409,SRSC 00175,LightCurve,2.0,2.6,SB21624_component_2449a,SB21624_RACS_1817+23,59234.10746528,59234.11795139,2021-01-20T02:34:45.000,2021-01-20T02:49:51,906.0,271.4569,21.4465,2.0,2.0,18:05:49.6,+21:26:47,1367.5,1.3,0.3,1.1,0.4,12.0,8.0,11.8,3.0,2.0,--,Detection,NoMeasurement,--,--,--,--,ASKAP,--,0.23,SRSCv1.0,SB21624_RACS_1817+23
4462,SRSC 00248,LightCurve,2.0,0.9,RACS_0913-50A_component_2071a,SB13800_RACS_0913-50A,58972.41628472,58972.42677083,2020-05-03T09:59:27.000,2020-05-03T10:14:33,906.0,142.6777,-53.5212,2.0,2.0,09:30:42.6,-53:31:16,887.5,4.5,0.03,4.74,0.08,15.3,13.0,164.1,0.1,0.2,--,Detection,NoMeasurement,--,--,--,--,ASKAP,--,0.51,SRSCv1.0,SB13800_RACS_0913-50A


As expected, this didn't decrease our sample size by a huge amount. Now, we don't want to have to process all of this data, as much of it is probably still false detections. What we can do instead is try and look for visible detections in Stokes V, which is a hallmark of some sort of flare/stellar activity. To do this, we need to generate a whole bunch of cutouts.

# Cutouts

Let's see if we can filter it so we only select image cubes that match our unique_obs_id. This code is adapted from Laura's code.

In [8]:
tap = TapPlus(url="https://casda.csiro.au/casda_vo_tools/tap")

In [9]:
obs_ids = list(source_obs_askap['unique_obs_id'])

In [32]:
# Coords of my source
source = sources[0]
source_coords = SkyCoord(source['Archival_ra']*u.deg, source['Archival_dec']*u.deg)
source_name = source['Simbad_ID']

# Directory where I'm going to save the cutouts I make
cutout_path = '../cutouts/'

AttributeError: 'str' object has no attribute 'deg'

In [11]:
# Set up the query for stokes V observations
image_type = 'V'
tap_qry = (F"SELECT * FROM ivoa.obscore where(pol_states = '/{image_type}/' and "
           F"dataproduct_subtype = 'cont.restored.t0') "
           F"AND 1 = CONTAINS(POINT('ICRS',{source_coords.ra.deg},{source_coords.dec.deg}),s_region)")

In [12]:
# Run the query
job = tap.launch_job_async(tap_qry)
r = job.get_results()
r = Casda.filter_out_unreleased(r)

INFO: Query finished. [astroquery.utils.tap.core]


In [34]:
r

dataproduct_type,calib_level,obs_collection,obs_id,obs_publisher_did,access_url,access_format,access_estsize,target_name,s_ra,s_dec,s_fov,s_region,s_resolution,s_xel1,s_xel2,t_min,t_max,t_exptime,t_resolution,t_xel,em_min,em_max,em_res_power,em_xel,o_ucd,pol_states,pol_xel,facility_name,instrument_name,dataproduct_subtype,em_ucd,em_unit,em_resolution,s_resolution_min,s_resolution_max,s_ucd,s_unit,obs_release_date,quality_level,thumbnail_id,filename
,,,,,,,kbyte,,deg,deg,deg,,arcsec,,,d,d,s,s,,m,m,,,,,,,,,,,m,arcsec,arcsec,,,,,,
str255,int32,str255,str255,str255,str2000,str255,int64,str255,float64,float64,float64,str200,float64,int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,int64,str255,str255,int64,str255,str255,str255,str255,str255,float64,float64,float64,str255,str255,str24,str15,int64,str1000
cube,2,The Rapid ASKAP Continuum Survey,ASKAP-21972,cube-97256,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=cube-97256,application/x-votable+xml;content=datalink,548112,,175.99819451458,-51.201187342649,43.306706790130384,POLYGON ICRS 182.16066842678 -54.04816810630902 169.84743879455996 -54.05752281588198 170.5867642266001 -48.06989868210401 181.3983882429701 -48.06168241513999,9.19999999999998,12986,10805,59239.80642476852,59239.81689930556,905.0,--,1,0.21922826338063595,0.2192282635409503,1367490000.0,1,phot.flux.density;phys.polarisation,/V/,1,ASKAP,,cont.restored.t0,em.freq,m,1.6031434513230636e-10,8.499999999999995,9.19999999999998,pos.eq,deg,2023-04-26T07:34:59.147Z,GOOD,--,image.v.RACS_1143-51.SB21972.cont.taylor.0.restored.conv.fits
cube,2,The Rapid ASKAP Continuum Survey,ASKAP-36393,cube-309509,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=cube-309509,application/x-votable+xml;content=datalink,867415,,175.93763922582,-51.132814884532,38.551245063205904,POLYGON ICRS 181.7465990556598 -53.813092044610016 170.13218873015012 -53.815922950989 170.79433772219986 -48.197334240254975 181.07755897209006 -48.19482693322199,7.300000000000007,16429,13516,59602.84760069445,59602.85807523148,905.748,--,1,0.18108986336942842,0.1810898634788159,1655490000.0,1,phot.flux.density;phys.polarisation,/V/,1,ASKAP,,cont.restored.t0,em.freq,m,1.093874713031795e-10,6.199999999999992,7.300000000000007,pos.eq,deg,2024-04-11T00:41:19.674Z,GOOD,--,image.v.RACS_1143-51.SB36393.cont.taylor.0.restored.conv.fits
cube,2,The Rapid ASKAP Continuum Survey,ASKAP-39134,cube-415775,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=cube-415775,application/x-votable+xml;content=datalink,558060,,175.39110121604,-50.048338650615,68.89426370113307,POLYGON ICRS 182.50218889972 -53.982014043306 168.28118099215 -53.982809486493 169.40330270174 -45.74732342128 181.37785745806 -45.746653050067,13.30000000000002,12011,11894,59677.62546527778,59677.635824074074,895.795,--,1,0.3377977442375203,0.3377977446181413,887491000.0,1,phot.flux.density;phys.polarisation,/V/,1,ASKAP,,cont.restored.t0,em.freq,m,3.8062103469016506e-10,11.7,13.30000000000002,pos.eq,deg,2025-02-11T03:17:52.747Z,GOOD,--,image.v.RACS_1141-50.SB39134.cont.taylor.0.restored.conv.fits
cube,2,The Rapid ASKAP Continuum Survey,ASKAP-40788,cube-436985,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=cube-436985,application/x-votable+xml;content=datalink,560737,,175.31913060238,-50.087903796236,69.22482287031184,POLYGON ICRS 182.42272047799 -54.053600365047 168.20364753783 -54.045580125962 169.34624978737 -45.756681203811 181.30260075677 -45.763430372317,12.999999999999996,11990,11972,59719.494099537034,59719.504574074075,905.748,--,1,0.3377977442375203,0.3377977446181413,887491000.0,1,phot.flux.density;phys.polarisation,/V/,1,ASKAP,,cont.restored.t0,em.freq,m,3.8062103469016506e-10,11.50000000000002,12.999999999999996,pos.eq,deg,2025-02-11T05:04:34.374Z,GOOD,--,image.v.RACS_1141-50.SB40788.cont.taylor.0.restored.conv.fits
cube,2,The Rapid ASKAP Continuum Survey,ASKAP-40792,cube-436965,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=cube-436

Note this is a list of all observations of this source, not all detections of this source. To get just our detections, we need to use our list of unique observation ids.

In [13]:
# Format our ids like the obs_id line in the table
sb_ids = [f"ASKAP-{obs_id.removeprefix('SB').split('_')[0]}" for obs_id in obs_ids]

In [14]:
# Select only the matches
r_detections = r[np.isin(r['obs_id'], sb_ids)]

# Filter our any other data types that might be in here
filename = np.array(r_detections['filename'], dtype=str)
collection = np.array(r_detections['obs_collection'], dtype=str)

mask = (
    (np.char.find(filename, 'raw') == -1) &
    (np.char.find(filename, 'alt') == -1) &
    (np.char.find(filename, 'highres') == -1) &
    (np.char.find(filename, 'iqr') == -1) &
    (np.char.find(collection, 'BETA') == -1)
)

r_detections = r_detections[mask]

In [82]:
r_detections

dataproduct_type,calib_level,obs_collection,obs_id,obs_publisher_did,access_url,access_format,access_estsize,target_name,s_ra,s_dec,s_fov,s_region,s_resolution,s_xel1,s_xel2,t_min,t_max,t_exptime,t_resolution,t_xel,em_min,em_max,em_res_power,em_xel,o_ucd,pol_states,pol_xel,facility_name,instrument_name,dataproduct_subtype,em_ucd,em_unit,em_resolution,s_resolution_min,s_resolution_max,s_ucd,s_unit,obs_release_date,quality_level,thumbnail_id,filename
,,,,,,,kbyte,,deg,deg,deg,,arcsec,,,d,d,s,s,,m,m,,,,,,,,,,,m,arcsec,arcsec,,,,,,
str255,int32,str255,str255,str255,str2000,str255,int64,str255,float64,float64,float64,str200,float64,int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,int64,str255,str255,int64,str255,str255,str255,str255,str255,float64,float64,float64,str255,str255,str24,str15,int64,str1000
cube,2,The Rapid ASKAP Continuum Survey,ASKAP-21972,cube-97256,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=cube-97256,application/x-votable+xml;content=datalink,548112,,175.99819451458,-51.201187342649,43.306706790130384,POLYGON ICRS 182.16066842678 -54.04816810630902 169.84743879455996 -54.05752281588198 170.5867642266001 -48.06989868210401 181.3983882429701 -48.06168241513999,9.19999999999998,12986,10805,59239.80642476852,59239.81689930556,905.0,--,1,0.21922826338063595,0.2192282635409503,1367490000.0,1,phot.flux.density;phys.polarisation,/V/,1,ASKAP,,cont.restored.t0,em.freq,m,1.6031434513230636e-10,8.499999999999995,9.19999999999998,pos.eq,deg,2023-04-26T07:34:59.147Z,GOOD,--,image.v.RACS_1143-51.SB21972.cont.taylor.0.restored.conv.fits
cube,2,WALLABY,ASKAP-50099,cube-143260,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=cube-143260,application/x-votable+xml;content=datalink,828684,,175.71475247046,-53.69357499905,65.47501614569363,POLYGON ICRS 183.27815223927 -57.52294205753172 168.15144578252992 -57.52300127781001 169.4710250649102 -49.46959427818999 181.95839533727002 -49.469545346063974,8.799999999999985,14578,14552,60084.40834722222,60084.741726851855,28804.0,--,1,0.21922826338063595,0.2192282635409503,1367490000.0,1,phot.flux.density;phys.polarisation,/V/,1,ASKAP,,cont.restored.t0,em.freq,m,1.6031434513230636e-10,7.700000000000004,8.799999999999985,pos.eq,deg,2023-07-03T02:17:44.273Z,GOOD,--,image.v.WALLABY_1142-53A.SB50099.cont.taylor.0.restored.conv.fits
cube,2,VAST,ASKAP-64834,cube-390276,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=cube-390276,application/x-votable+xml;content=datalink,768870,,175.38515383189,-50.102158319038,94.91986801275338,POLYGON ICRS 183.85399567287 -54.696407757043 166.91642760662 -54.696486785186 168.47171518781 -45.003221023467 182.29849131824 -45.003156432518,13.600000000000009,14044,14015,60538.281369212964,60538.28988773148,736.543,--,1,0.3377977442375203,0.3377977446181413,887491000.0,1,phot.flux.density;phys.polarisation,/V/,1,ASKAP,,cont.restored.t0,em.freq,m,3.8062103469016506e-10,11.50000000000002,13.600000000000009,pos.eq,deg,2024-08-29T01:07:35.487Z,UNCERTAIN,--,image.v.VAST_1141-50.SB64834.cont.taylor.0.restored.conv.fits
cube,2,WALLABY,ASKAP-72938,cube-473381,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=cube-473381,application/x-votable+xml;content=datalink,837757,,176.49515328807,-54.141390987528,66.19177909619597,POLYGON ICRS 184.1956360916 -57.988534594829 168.7972651222 -57.990171469874 170.16571289259 -49.888941627512004 182.82223434831 -49.887594917424,8.399999999999988,14648,14641,60781.48491319444,60781.81840856482,28814.7,--,1,0.21922826338063595,0.2192282635409503,1367490000.0,1,phot.flux.density;phys.polarisation,/V/,1,ASKAP,,cont.restored.t0,em.freq,m,1.6031434513230636e-10,7.400000000000016,8.399999999999988,pos.eq,deg,2025-05-13T01:14:58.512Z,GOOD,--,image.v.WALLABY_1142-53B.SB72938.cont.taylor.0.restored.conv.fits


So now we are left with a table of 4 image cubes, conveniently the same length as our number of detections for this first source! Next we just need to download these image cubes

In [16]:
# Specify cutout width
cutout_width = 4 * u.arcmin

print('Staging the cutouts files')
clean_url_list = []
url_list = Casda.cutout(r_detections, coordinates = source_coords, radius = cutout_width)

# Avoid downloading checksum files
for row in url_list:
    if 'checksum' not in row:
        clean_url_list.append(row)

# Download the data
filelist = Casda.download_files(clean_url_list, savedir=cutout_path)

Staging the cutouts files


While this was successful, it did leave us with some pretty gnarly file names. We need to find a way to map file names to both the source and observation so we can find that data if we need it. Ideally, we can name it something like {SRSC_XXXX_SURVEY_XXXX.fits}.

Before we do that, I want to check just how many cutouts we will have per query to know if I need to segment our cutout calls so we don't make one massive request to CASDA.

In [49]:
for source in sources:
    
    # Source info
    source_coords = SkyCoord(source['Archival_ra']*u.deg, source['Archival_dec']*u.deg)
    source_name = source['Simbad_ID']
    source_name = re.sub(r'[^a-z0-9_-]', '', source_name.lower().replace(" ", "_"))
    
    # Set up the query for stokes V observations
    image_type = 'V'
    tap_qry = (F"SELECT * FROM ivoa.obscore where(pol_states = '/{image_type}/' and "
               F"dataproduct_subtype = 'cont.restored.t0') "
               F"AND 1 = CONTAINS(POINT('ICRS',{source_coords.ra.deg},{source_coords.dec.deg}),s_region)")
    
    # Run the query
    job = tap.launch_job_async(tap_qry)
    r = job.get_results()
    r = Casda.filter_out_unreleased(r)
    
    # Format our ids like the obs_id line in the table
    sb_ids = [f"ASKAP-{obs_id.removeprefix('SB').split('_')[0]}" for obs_id in obs_ids]
    
    # Select only the matches
    r_detections = r[np.isin(r['obs_id'], sb_ids)]
    
    # Filter our any other data types that might be in here
    filename = np.array(r_detections['filename'], dtype=str)
    collection = np.array(r_detections['obs_collection'], dtype=str)
    
    mask = (
        (np.char.find(filename, 'raw') == -1) &
        (np.char.find(filename, 'alt') == -1) &
        (np.char.find(filename, 'highres') == -1) &
        (np.char.find(filename, 'iqr') == -1) &
        (np.char.find(collection, 'BETA') == -1)
    )
    
    r_detections = r_detections[mask]
    
    # Print our results
    print(f'{source_name}: {len(r_detections)} Cutouts to download')  

INFO: Query finished. [astroquery.utils.tap.core]
hd_102982: 4 Cutouts to download
INFO: Query finished. [astroquery.utils.tap.core]
v_v376_cep: 0 Cutouts to download
INFO: Query finished. [astroquery.utils.tap.core]
v_v815_her: 1 Cutouts to download
INFO: Query finished. [astroquery.utils.tap.core]
hd_165590a: 2 Cutouts to download
INFO: Query finished. [astroquery.utils.tap.core]
hd__82533: 31 Cutouts to download
INFO: Query finished. [astroquery.utils.tap.core]
hd_217344: 42 Cutouts to download
INFO: Query finished. [astroquery.utils.tap.core]
hd___8558: 7 Cutouts to download
INFO: Query finished. [astroquery.utils.tap.core]
hd_200244: 1 Cutouts to download
INFO: Query finished. [astroquery.utils.tap.core]
hd_162186: 1 Cutouts to download
INFO: Query finished. [astroquery.utils.tap.core]
hd_104919: 5 Cutouts to download
INFO: Query finished. [astroquery.utils.tap.core]
hd__78644: 4 Cutouts to download
INFO: Query finished. [astroquery.utils.tap.core]
hd_156097: 1 Cutouts to download

So we will need to segment them, we can do this with Laura's code

In [50]:
r_df = r_detections.to_pandas()

In [51]:
    # You really can't download 100+ cutouts all at once. So I split
    # this into smaller batches. Here I'm doing batches of 5
    url_loop_length = 5
    
    # The size of the cutout. This will be the length
    # of each edge of the square cutout
    cutout_width = 4 * u.arcmin
    
    print('Staging the noise files')
    clean_url_list = []
    r_filenames = np.array(r_df['filename'])
    # I'm keeping it to one loop here, for demonstration purposes
    for url_ in np.arange((len(r_filenames) // url_loop_length) + 1)[:2]:
        url_filenames = r_filenames[url_ * url_loop_length: (url_ + 1) * url_loop_length]
        print(url_filenames)
        url_r_df = r_df[r_df['filename'].isin(url_filenames)]
        url_list = Casda.cutout(Table.from_pandas(url_r_df),
                                coordinates=source_coords, radius=cutout_width)
        # I find it annoying to download the checksum files
        # with the files I actually want
        for row in url_list:
            if 'checksum' not in row:
                clean_url_list.append(row)
                
        print('Downloading the files')
        # Now we can download that batch of files
        filelist = Casda.download_files(clean_url_list,
                                        savedir=cutout_path)
        
        # Here I rename the downloaded files. They come out with
        # quite nasty filenames that don't give you wany real info.
        # So I give them back their original names
        for cf, cutf in enumerate(filelist):
            cube_id = 'cube-' + (cutf.split('-imagecube-')[-1]).split('.fits')[0]
        
            source_row = r_df[r_df['obs_publisher_did'] == cube_id]
        
            og_name = source_row['filename'].item()
        
            new_name = F'{cutout_path}{source_name}_{og_name}'
        
            os.rename(cutf, new_name)
        # Changing the file names does mean that you'll have to do
        # checks yourself to avoid downloading the same cutout twice
        clean_url_list = []

Staging the noise files
['image.v.EMU_0649-51.SB62517.cont.taylor.0.restored.conv.fits'
 'image.v.RACS_0649-51.SB67955.cont.taylor.0.restored.conv.fits'
 'image.v.EMU_0721-51.SB76195.cont.taylor.0.restored.conv.fits']


This is working well now, lets move this over to a script that we can run and download all of our cutouts